In [64]:
# ==========================================
# 1. CONFIGURATION & GLOBAL DATA LOAD
# ==========================================
import os
import re
import json
import numpy as np
import pandas as pd
import open3d as o3d
import matplotlib.pyplot as plt

# File paths and naming
EXPERIMENT = "test_8_simulation2"
DATA_FOLDER = "TH0011AV"           # Folder containing the point clouds and listed in the CSV
CAD_MODEL_DIR = "TH0011AV"      # Folder containing the .stl feature models (e.g. workpiece3, workpiece31)

ENABLE_VISUALIZATION = True

USE_GROUND_TRUTH = True
GT_CSV_PATH = f"processed_data/{EXPERIMENT}/metadata.csv"

CSV_PATH = "../../pointnet_pytorch_reflective/data/4_simulation/inference_results_pointnet2_moe.csv"

WORKPIECE_PATH = f"workpiece/{CAD_MODEL_DIR}/workpiece.stl"
POSES_PATH = f"viewpoints_candidate/testing_data/{EXPERIMENT}/{DATA_FOLDER}"

# The EXACT 40k points saved from 1_3_viewpoint_generation_manual
GLOBAL_PCD_PATH = f"viewpoints_candidate/testing_data/{EXPERIMENT}/{DATA_FOLDER}/pcd_all.pcd"
GLOBAL_COVERED_JSON = f"viewpoints_candidate/testing_data/{EXPERIMENT}/{DATA_FOLDER}/covered_indices.json"

# Optimization Settings
OPTIMIZATION_METHOD = "GRASP"  # "GREEDY" or "GRASP"
GRASP_ITERATIONS = 2          # Number of sequences to generate
RCL_SIZE = 5                   # Top N candidates to pick randomly from (Cardinality-based RCL)

# Utility weights & Discretization
ALPHA = 0.7 # Coverability
BETA = 0.3 # Confidence
GAMMA = 0.5  # Submodular decay factor for Coverability
TOTAL_STEPS = 1  # Number of viewpoints to select
DISTANCE_THRESHOLD = 1.0  # mm
AXIS_SIZE = 20.0
NUM_BINS = 4

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def make_arrow(direction='x', size=50.0, color=(0.6, 0.6, 0.6)):
    arrow = o3d.geometry.TriangleMesh.create_arrow(
        cylinder_radius=size * 0.05,
        cone_radius=size * 0.15,
        cylinder_height=size * 0.80,
        cone_height=size * 0.20,
        resolution=20,
    )
    if direction == 'x':
        R_align = arrow.get_rotation_matrix_from_xyz((0, np.pi / 2, 0))
        arrow.rotate(R_align, center=(0, 0, 0))
    arrow.paint_uniform_color(list(color))
    arrow.compute_vertex_normals()
    return arrow

def make_xz_arrows(transform=None, size=50.0, color=(0.6, 0.6, 0.6)):
    frame = make_arrow('x', size=size, color=color) + make_arrow('z', size=size, color=color)
    if transform is not None:
        frame.transform(transform)
    return frame

def load_viewpoint_poses_dict(folder_path):
    def extract_number(filename):
        match = re.search(r'viewpoint_pose_(\d+)\.npy', filename)
        return int(match.group(1)) if match else -1
    
    npy_files = [f for f in os.listdir(folder_path) if f.endswith('.npy')]
    poses = {}
    for f in npy_files:
        idx = extract_number(f)
        if idx != -1:
            poses[idx] = np.load(os.path.join(folder_path, f))
    return poses

# ==========================================
# 3. LOAD GLOBAL DATA & MULTIPLE FEATURES
# ==========================================
print("Loading Global 40k Point Cloud...")
pcd_all = o3d.io.read_point_cloud(GLOBAL_PCD_PATH) if os.path.exists(GLOBAL_PCD_PATH) else o3d.geometry.PointCloud()
pcd_all.paint_uniform_color([0.6, 0.6, 0.6])

print("Loading Global Viewpoint Visibilities...")
if os.path.exists(GLOBAL_COVERED_JSON):
    with open(GLOBAL_COVERED_JSON, "r") as f:
        global_visibility_dict = json.load(f)
else:
    global_visibility_dict = {}

poses_dict = load_viewpoint_poses_dict(POSES_PATH)

# Load Inference Results CSV and parse Filename column
if USE_GROUND_TRUTH:
    df_inference = pd.read_csv(GT_CSV_PATH)
    df_inference.rename(columns={'filename': 'Filename', 'chamfer_value': 'Predicted_CD'}, inplace=True)
    print(f"\n--- USING GROUND TRUTH DATA FROM {GT_CSV_PATH} ---")
else:
    df_inference = pd.read_csv(CSV_PATH)
    print(f"\n--- USING PREDICTED DATA FROM {CSV_PATH} ---")
df_inference[['Parsed_Folder', 'Viewpoint', 'Feature']] = df_inference['Filename'].str.extract(r'(.*?)/viewpoint_simulated_(\d+)_(.*?)\.pcd')
df_inference['Viewpoint'] = df_inference['Viewpoint'].astype(int)

# Filter for the requested workpiece folder
df_chamfer = df_inference[df_inference['Parsed_Folder'] == DATA_FOLDER].copy()
df_chamfer.rename(columns={'Predicted_CD': 'Chamfer_Distance_mm'}, inplace=True)

features = df_chamfer['Feature'].unique()
print(f"\nFound {len(features)} unique features to optimize for {DATA_FOLDER}: {features}")

feature_indices = {}
all_features_indices = set()

for feat in features:
    feat_path = f"workpiece/{CAD_MODEL_DIR}/{feat}.stl"
    if os.path.exists(feat_path):
        f_mesh = o3d.io.read_triangle_mesh(feat_path)
        f_pcd = f_mesh.sample_points_poisson_disk(number_of_points=5000)
        
        dists = np.asarray(pcd_all.compute_point_cloud_distance(f_pcd))
        idx_set = set(np.where(dists < DISTANCE_THRESHOLD)[0])
        feature_indices[feat] = idx_set
        all_features_indices.update(idx_set)
        
        print(f"  - {feat}: {len(idx_set)} matching points in 40k cloud")
    else:
        print(f"  - WARNING: {feat_path} not found!")

print(f"\nTotal target points across all features: {len(all_features_indices)}")


Loading Global 40k Point Cloud...
Loading Global Viewpoint Visibilities...

--- USING GROUND TRUTH DATA FROM processed_data/test_8_simulation2/metadata.csv ---

Found 2 unique features to optimize for TH0011AV: ['surface0' 'surface1']
  - surface0: 5498 matching points in 40k cloud
  - surface1: 8109 matching points in 40k cloud

Total target points across all features: 13092


In [65]:
# ==========================================
# 4. OPTIMIZATION RUN (GREEDY / GRASP)
# ==========================================
import pandas as pd
import numpy as np
import random

if OPTIMIZATION_METHOD == "GREEDY":
    num_iterations = 1
    rcl_size = 1
else:
    num_iterations = GRASP_ITERATIONS
    rcl_size = RCL_SIZE

if len(all_features_indices) == 0:
    print("WARNING: all_features_indices is empty. Make sure you loaded the data correctly.")

print(f"Starting {OPTIMIZATION_METHOD} Optimization for {TOTAL_STEPS} steps...")
if OPTIMIZATION_METHOD == "GRASP":
    print(f"Running {num_iterations} iterations with RCL Size = {rcl_size} (Cardinality)...\n")

# Global tracking for the absolute best sequence found
best_overall_sequence = []
best_overall_score = -1.0
best_overall_point_counts = None
best_step_to_covered = {}
best_all_step_results = []

# --- PRECOMPUTE STATIC CHAMFER DATA ---
static_viewpoint_data = {}
for v_idx, group in df_chamfer.groupby('Viewpoint'):
    v_idx = int(v_idx)
    if v_idx not in poses_dict or pd.isna(group['Chamfer_Distance_mm'].iloc[0]): continue
    camera_visible_40k = set(global_visibility_dict.get(str(v_idx), []))
    visible_target_indices = camera_visible_40k & all_features_indices
    if len(visible_target_indices) == 0: continue
    sum_weighted_chamfer = 0.0
    sum_points = 0
    for _, row in group.iterrows():
        feature = row['Feature']
        chamfer_dist = row['Chamfer_Distance_mm']
        if chamfer_dist < 0 or feature not in feature_indices: continue
        P_vj = len(camera_visible_40k & feature_indices[feature])
        sum_weighted_chamfer += (P_vj * chamfer_dist)
        sum_points += P_vj
    if sum_points == 0: continue
    weighted_chamfer = sum_weighted_chamfer / sum_points
    static_viewpoint_data[v_idx] = {
        'weighted_chamfer': weighted_chamfer,
        'visible_target_indices': visible_target_indices,
        'idx_arr': list(visible_target_indices)
    }

for iteration in range(1, num_iterations + 1):
    
    # State is completely reset at the start of every sequence generation
    point_coverage_counts = np.zeros(len(pcd_all.points))
    selected_viewpoints = []
    step_to_covered_indices = {}
    all_step_results = []
    
    cumulative_utility = 0.0
    
    for k in range(1, TOTAL_STEPS + 1):
        db_records = []
        
        for v_idx, data in static_viewpoint_data.items():
            if v_idx in selected_viewpoints:
                continue
                
            idx_arr = data['idx_arr']
            c_vals = point_coverage_counts[idx_arr]
            submodular_sum = np.sum(GAMMA ** c_vals)
            coverability = submodular_sum / len(all_features_indices)
            
            # Vectorized newly covered calculation
            newly_covered_arr = np.array(idx_arr)[c_vals == 0]
            newly_covered = newly_covered_arr.tolist()
            
            db_records.append({
                'Step': k,
                'Viewpoint': v_idx,
                'Coverability': coverability,
                'Chamfer_Distance': data['weighted_chamfer'],
                '_visible_target_indices': data['visible_target_indices'],
                '_new_points_set': newly_covered
            })
        
        df_step = pd.DataFrame(db_records)
        if df_step.empty:
            break
            
        # --- Normalize Metrics ---
        min_cov = df_step['Coverability'].min()
        max_cov = df_step['Coverability'].max()
        df_step['Norm_Coverability'] = (df_step['Coverability'] - min_cov) / (max_cov - min_cov) if max_cov > min_cov else 1.0
        
        # --- Uncertainty & Confidence ---
        df_step['Uncertainty'] = (1.0 * df_step['Norm_Coverability']) + (1.0 * df_step['Chamfer_Distance'])
        min_uncert = df_step['Uncertainty'].min()
        max_uncert = df_step['Uncertainty'].max()
        df_step['Confidence'] = 1.0 - ((df_step['Uncertainty'] - min_uncert) / (max_uncert - min_uncert)) if max_uncert > min_uncert else 1.0
        
        # --- Utility Score ---
        df_step['Information_Gain'] = df_step['Norm_Coverability']
        df_step['Utility_Score'] = ALPHA * df_step['Information_Gain'] + BETA * df_step['Confidence']
        
        # Rank Candidates
        df_step = df_step.sort_values(by='Utility_Score', ascending=False).reset_index(drop=True)
        df_step['Rank'] = df_step.index + 1
        
        # --------------------------------------------------
        # RCL SELECTION (CARDINALITY)
        # --------------------------------------------------
        actual_rcl_size = min(rcl_size, len(df_step))
        rcl = df_step.head(actual_rcl_size)
        
        # Pick completely randomly from the RCL
        chosen_idx = random.randint(0, actual_rcl_size - 1)
        best_row = rcl.iloc[chosen_idx]
        
        best_viewpoint = best_row['Viewpoint']
        best_visible_indices = best_row['_visible_target_indices']
        best_new_points = best_row['_new_points_set']
        
        # Track total utility across the sequence
        cumulative_utility += best_row['Utility_Score']
        
        selected_viewpoints.append(int(best_viewpoint))
        all_step_results.append(df_step.drop(columns=['_visible_target_indices', '_new_points_set']))
        step_to_covered_indices[k] = best_new_points
        
        # Update state counts for the next step in this sequence
        for idx in best_visible_indices:
            point_coverage_counts[idx] += 1
            
    # After generating a full sequence, check if it's the best one we've seen globally
    if cumulative_utility > best_overall_score:
        best_overall_score = cumulative_utility
        best_overall_sequence = selected_viewpoints
        best_overall_point_counts = point_coverage_counts.copy()
        best_step_to_covered = step_to_covered_indices
        best_all_step_results = all_step_results
        
    if OPTIMIZATION_METHOD == "GRASP":
        print(f"Iteration {iteration}/{num_iterations} | Current Utility: {cumulative_utility:.4f} | Best Sum Utility: {best_overall_score:.4f}")

# Expose the best iteration globally so visualization blocks can map colors correctly
point_coverage_counts = best_overall_point_counts
selected_viewpoints = best_overall_sequence
step_to_covered_indices = best_step_to_covered
all_step_results = best_all_step_results

print(f"\n{'='*40}")
print(f"OPTIMIZATION COMPLETE ({OPTIMIZATION_METHOD})")
print(f"\n--- PARAMETERS USED ---")
print(f"Alpha (Coverability Weight): {ALPHA}")
print(f"Beta (Confidence Weight): {BETA}")
print(f"Total Steps: {TOTAL_STEPS}")
print(f"Using Ground Truth Data: {USE_GROUND_TRUTH}")
if OPTIMIZATION_METHOD == 'GRASP':
    print(f"GRASP Iterations: {GRASP_ITERATIONS}")
    print(f"GRASP RCL Size (Top-N): {RCL_SIZE}")
print("-----------------------\n")
print(f"Total points covered at least once: {np.sum(point_coverage_counts > 0)} / {len(all_features_indices)}")
print(f"Best Sequence Sum Utility: {best_overall_score:.4f}")
print(f"Best Viewpoint Sequence: {selected_viewpoints}\n")
print(f"Breakdown of the Best Sequence:")
for k, (v_idx, df_step) in enumerate(zip(selected_viewpoints, all_step_results), 1):
    row = df_step[df_step['Viewpoint'] == v_idx].iloc[0]
    cov = row['Coverability']
    conf = row['Confidence']
    util = row['Utility_Score']
    print(f"  Step {k} -> Viewpoint {int(v_idx)}: Utility = {util:.4f} | Coverability = {cov:.4f} | Confidence = {conf:.4f}")
print("\n")

if OPTIMIZATION_METHOD == "GREEDY":
    for k, df_step in enumerate(all_step_results, 1):
        print(f"--- STEP {k} Top Candidates ---")
        display(df_step.head(5))
else:
    print("The above sequence is the best sequence found across all GRASP iterations!")


Starting GRASP Optimization for 1 steps...
Running 2 iterations with RCL Size = 5 (Cardinality)...

Iteration 2/2 | Best Sum Utility: 0.7247

OPTIMIZATION COMPLETE (GRASP)

--- PARAMETERS USED ---
Alpha (Coverability Weight): 0.7
Beta (Confidence Weight): 0.3
Total Steps: 1
Using Ground Truth Data: True
GRASP Iterations: 2
GRASP RCL Size (Top-N): 5
-----------------------

Total points covered at least once: 11922 / 13092
Best Sequence Sum Utility: 0.7247
Best Viewpoint Sequence: [7]

Breakdown of the Best Sequence:
  Step 1 -> Viewpoint 7: Utility = 0.7247 | Coverability = 0.9106 | Confidence = 0.1337


The above sequence is the best sequence found across all GRASP iterations!


In [66]:
# ==========================================
# 3. VISUALIZATION COVERAGE PER VIEWPOINT
 # ==========================================
import copy
import matplotlib.pyplot as plt

# Create a copy of pcd_all to colorize
vis_pcd = copy.deepcopy(pcd_all)

# Default color (light gray) for unseen points
colors = np.ones((len(vis_pcd.points), 3)) * 0.8

# Generate distinct colors for each step (e.g. from tab10 colormap)
cmap = plt.get_cmap("tab10")

print("Point Colors:")
for k in range(1, TOTAL_STEPS + 1):
    if k in step_to_covered_indices:
        step_color = cmap(k - 1)[:3]  # RGB from colormap
        
        # Color text for the print statement using ANSI escape codes
        r, g, b = [int(c * 255) for c in step_color]
        colored_text = f"\033[38;2;{r};{g};{b}mStep {k}\033[0m"
        print(f"{colored_text} covered {len(step_to_covered_indices[k])} new points.")
        
        indices = list(step_to_covered_indices[k])
        colors[indices] = step_color

vis_pcd.colors = o3d.utility.Vector3dVector(colors)

# Draw geometries
print("\nOpening Open3D visualization window...")
if ENABLE_VISUALIZATION:
    o3d.visualization.draw_geometries(
        [vis_pcd], 
        window_name="Step-by-Step Coverage",
        width=1024, height=768,
        front=[0, 0, 1], lookat=[0, 0, 0], up=[0, 1, 0], zoom=1.0
    )


Point Colors:
Step 1 covered 11922 new points.

Opening Open3D visualization window...


In [ ]:
# ==========================================
# 4. EXTRA: VISUALIZE ONLY DETECTED POINTS
# ==========================================
import open3d as o3d
import numpy as np

print("Filtering out unseen (grey) points...")

# Collect all indices of points covered in ANY step
all_covered_indices = set()
for step, indices in step_to_covered_indices.items():
    all_covered_indices.update(indices)

# Use Open3D's select_by_index to extract ONLY the colored points
covered_pcd = vis_pcd.select_by_index(list(all_covered_indices))

print(f"Total points shown: {len(covered_pcd.points)}")

# Draw geometries
print("\nOpening Open3D visualization window (Detected Points Only)...")
if ENABLE_VISUALIZATION:
    o3d.visualization.draw_geometries(
        [covered_pcd], 
        window_name="Detected Points Only",
        width=1024, height=768,
        front=[0, 0, 1], lookat=[0, 0, 0], up=[0, 1, 0], zoom=1.0
    )


Filtering out unseen (grey) points...
Total points shown: 10119

Opening Open3D visualization window (Detected Points Only)...


In [67]:
# ==========================================
# 5. VISUALIZE SELECTED GRASP POSES
# ==========================================
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt

print("Generating visualization of the entire workpiece and camera poses...")

# We use the full workpiece (vis_pcd) instead of filtering it!
geometries = [vis_pcd]

cmap = plt.get_cmap("tab10")

print("\n--- Selected Viewpoints Summary ---")
# For each step, create the camera pose arrows matching the step color
for k, v_idx in enumerate(selected_viewpoints, 1):
    step_color = cmap(k - 1)[:3]
    
    # Print the summary
    r, g, b = [int(c * 255) for c in step_color]
    colored_text = f"\033[38;2;{r};{g};{b}mStep {k}\033[0m"
    print(f"{colored_text}: Viewpoint {v_idx}")
    
    # Add the colored pose arrows to the visualization
    if v_idx in poses_dict:
        pose_arrows = make_xz_arrows(transform=poses_dict[v_idx], size=AXIS_SIZE, color=step_color)
        geometries.append(pose_arrows)

print("\nOpening Open3D visualization window...")

if ENABLE_VISUALIZATION:
    o3d.visualization.draw_geometries(
        geometries, 
        window_name="Entire Workpiece and Camera Poses",
        width=1024, height=768,
        front=[0, 0, 1], lookat=[0, 0, 0], up=[0, 1, 0], zoom=1.0
    )


Generating visualization of the entire workpiece and camera poses...

--- Selected Viewpoints Summary ---
Step 1: Viewpoint 7

Opening Open3D visualization window...


In [13]:
# ==========================================
# 6. STEP-BY-STEP OBSERVATION FREQUENCY
# ==========================================
import copy
import matplotlib.pyplot as plt
import open3d as o3d
import numpy as np

print("Generating progressive observation frequency visualizations...")
cmap = plt.get_cmap("tab10")

# We will simulate the frequency accumulation step-by-step
current_counts = np.zeros(len(pcd_all.points))

for k, v_idx in enumerate(selected_viewpoints, 1):
    print(f"\n{'='*40}")
    print(f" STEP {k} FREQUENCY STATE (After adding Viewpoint {v_idx})")
    print(f"{'='*40}")
    
    # 1. Fetch the exact points seen by this specific viewpoint
    camera_visible_40k = set(global_visibility_dict.get(str(v_idx), []))
    visible_target_indices = camera_visible_40k & all_features_indices
    
    # 2. Add them to our cumulative counts
    for idx in visible_target_indices:
        current_counts[idx] += 1
        
    # 3. Create point cloud for this step
    freq_pcd = copy.deepcopy(pcd_all)
    colors = np.ones((len(freq_pcd.points), 3)) * 0.8
    
    max_obs = int(np.max(current_counts))
    for count in range(1, max_obs + 1):
        indices = np.where(current_counts == count)[0]
        if len(indices) > 0:
            freq_color = cmap(count - 1)[:3]
            r, g, b = [int(c * 255) for c in freq_color]
            colored_text = f"\033[38;2;{r};{g};{b}mSeen {count} Time(s)\033[0m"
            print(f"{colored_text}: {len(indices)} points")
            colors[indices] = freq_color
            
    freq_pcd.colors = o3d.utility.Vector3dVector(colors)
    
    # 4. Filter out unseen points (0 count)
    observed_indices = np.where(current_counts > 0)[0]
    covered_freq_pcd = freq_pcd.select_by_index(list(observed_indices))
    
    print(f"Opening Open3D visualization window for Step {k}...")
    print("NOTE: Close the Open3D window to proceed to the next step!")
    
    o3d.visualization.draw_geometries(
        [covered_freq_pcd], 
        window_name=f"Observation Frequency (After Step {k})",
        width=1024, height=768,
        front=[0, 0, 1], lookat=[0, 0, 0], up=[0, 1, 0], zoom=1.0
    )


Generating progressive observation frequency visualizations...

 STEP 1 FREQUENCY STATE (After adding Viewpoint 326)
Seen 1 Time(s): 10893 points
Opening Open3D visualization window for Step 1...
NOTE: Close the Open3D window to proceed to the next step!

 STEP 2 FREQUENCY STATE (After adding Viewpoint 16)
Seen 1 Time(s): 3889 points
Seen 2 Time(s): 9126 points
Opening Open3D visualization window for Step 2...
NOTE: Close the Open3D window to proceed to the next step!

 STEP 3 FREQUENCY STATE (After adding Viewpoint 169)
Seen 1 Time(s): 1846 points
Seen 2 Time(s): 2433 points
Seen 3 Time(s): 8811 points
Opening Open3D visualization window for Step 3...
NOTE: Close the Open3D window to proceed to the next step!

 STEP 4 FREQUENCY STATE (After adding Viewpoint 3)
Seen 1 Time(s): 82 points
Seen 2 Time(s): 3271 points
Seen 3 Time(s): 1078 points
Seen 4 Time(s): 8659 points
Opening Open3D visualization window for Step 4...
NOTE: Close the Open3D window to proceed to the next step!


In [45]:
# ==========================================
# ENTIRE POSE AND CD VISUALIZATION
# ==========================================
import os
import re
import numpy as np
import pandas as pd
import open3d as o3d
import matplotlib.pyplot as plt

# 1. CONFIGURATION
EXPERIMENT = "test_8_simulation2"
DATA_FOLDER = "TH0011AV"
CAD_MODEL_DIR = "TH0011AV"
FEATURE_NAME = "surface1"

WORKPIECE_PATH = f"workpiece/{CAD_MODEL_DIR}/workpiece.stl"
FEATURE_PATH = f"workpiece/{CAD_MODEL_DIR}/{FEATURE_NAME}.stl"
POSES_PATH = f"viewpoints_candidate/testing_data/{EXPERIMENT}/{DATA_FOLDER}"
CSV_PATH = "../../pointnet_pytorch_reflective/data/4_simulation/inference_results_pointnet2_s0_extra.csv"
USE_GROUND_TRUTH = True
GT_CSV_PATH = f"processed_data/{EXPERIMENT}/metadata.csv"
CSV_PATH = "../../pointnet_pytorch_reflective/data/4_simulation/inference_results_pointnet2_s0_extra.csv"
AXIS_SIZE = 20.0
NUM_BINS = 4
DISTANCE_THRESHOLD = 1.0  # Threshold to remove overlapping points (mm)

# 2. HELPER FUNCTIONS
def make_arrow(direction='x', size=50.0, color=(0.6, 0.6, 0.6)):
    arrow = o3d.geometry.TriangleMesh.create_arrow(
        cylinder_radius=size * 0.05,
        cone_radius=size * 0.15,
        cylinder_height=size * 0.80,
        cone_height=size * 0.20,
        resolution=20,
    )
    if direction == 'x':
        R_align = arrow.get_rotation_matrix_from_xyz((0, np.pi / 2, 0))
        arrow.rotate(R_align, center=(0, 0, 0))
    arrow.paint_uniform_color(list(color))
    arrow.compute_vertex_normals()
    return arrow

def make_xz_arrows(transform=None, size=50.0, color=(0.6, 0.6, 0.6)):
    frame = make_arrow('x', size=size, color=color) + make_arrow('z', size=size, color=color)
    if transform is not None:
        frame.transform(transform)
    return frame

def visualize_viewpoints_colored(meshes, matrices, colors_per_frame, axis_size=50.0):
    if isinstance(matrices, np.ndarray) and matrices.ndim == 2:
        matrices = [matrices]

    geometries = list(meshes)
    geometries.append(make_xz_arrows(transform=None, size=axis_size, color=(0.9, 0.9, 0.9)))

    for i, (mat, color) in enumerate(zip(matrices, colors_per_frame)):
        geometries.append(make_xz_arrows(transform=mat, size=axis_size, color=color))

    print(f"\nVisualizing {len(matrices)} viewpoint(s) for feature '{FEATURE_NAME}'...")
    # If Open3D fails to show window in your environment, sometimes calling it without kwargs helps.
    ENABLE_VISUALIZATION = True
    if ENABLE_VISUALIZATION:
        o3d.visualization.draw_geometries(
            geometries,
            window_name="Feature Viewpoint Visualization",
            width=1024, height=768,
            front=[0, 0, 1], lookat=[0, 0, 0], up=[0, 1, 0], zoom=1.0
        )

def load_viewpoint_poses_dict(folder_path):
    def extract_number(filename):
        match = re.search(r'viewpoint_pose_(\d+)\.npy', filename)
        return int(match.group(1)) if match else -1
    
    npy_files = [f for f in os.listdir(folder_path) if f.endswith('.npy')]
    poses = {}
    for f in npy_files:
        idx = extract_number(f)
        if idx != -1:
            try:
                poses[idx] = np.load(os.path.join(folder_path, f))
            except Exception as e:
                print(f"Error loading {f}: {e}")
    return poses

# 3. LOAD DATA & COLOR PROCESSING
if os.path.exists(WORKPIECE_PATH):
    workpiece_mesh = o3d.io.read_triangle_mesh(WORKPIECE_PATH)
    workpiece_mesh.compute_vertex_normals()
    print("Sampling points from the whole workpiece mesh to create a PointCloud...")
    workpiece_pcd = workpiece_mesh.sample_points_poisson_disk(number_of_points=10000)
    workpiece_pcd.paint_uniform_color([0.6, 0.6, 0.6])  # Gray for the whole workpiece
else:
    print(f"Warning: {WORKPIECE_PATH} not found.")
    workpiece_pcd = o3d.geometry.PointCloud()

if os.path.exists(FEATURE_PATH):
    feature_mesh = o3d.io.read_triangle_mesh(FEATURE_PATH)
    feature_mesh.compute_vertex_normals()
    print("Sampling points from the feature mesh to create a PointCloud...")
    feature_pcd = feature_mesh.sample_points_poisson_disk(number_of_points=5000)
    feature_pcd.paint_uniform_color([1.0, 0.0, 0.0])  # Red for the feature
else:
    print(f"Warning: {FEATURE_PATH} not found.")
    feature_pcd = o3d.geometry.PointCloud()

# Remove overlapping points from the workpiece so the feature is perfectly clear
if len(workpiece_pcd.points) > 0 and len(feature_pcd.points) > 0:
    print("Removing points from the workpiece that overlap with the selected feature...")
    dists = workpiece_pcd.compute_point_cloud_distance(feature_pcd)
    dists = np.asarray(dists)
    non_feature_indices = np.where(dists > DISTANCE_THRESHOLD)[0]
    workpiece_pcd = workpiece_pcd.select_by_index(non_feature_indices)

poses_dict = load_viewpoint_poses_dict(POSES_PATH)
print(f"Loaded {len(poses_dict)} viewpoint poses from {POSES_PATH}")

if USE_GROUND_TRUTH:
    df_inference = pd.read_csv(GT_CSV_PATH)
    df_inference.rename(columns={'filename': 'Filename', 'chamfer_value': 'Predicted_CD'}, inplace=True)
else:
    df_inference = pd.read_csv(CSV_PATH)
df_inference[['Parsed_Folder', 'Viewpoint', 'Feature']] = df_inference['Filename'].str.extract(r'(.*?)/viewpoint_simulated_(\d+)_(.*?)\.pcd')
df_inference['Viewpoint'] = df_inference['Viewpoint'].astype(int)
df_chamfer = df_inference[df_inference['Parsed_Folder'] == DATA_FOLDER].copy()
df_chamfer.rename(columns={'Predicted_CD': 'Chamfer_Distance_mm'}, inplace=True)
df_feature = df_chamfer[df_chamfer['Feature'] == FEATURE_NAME].copy()
if df_feature.empty:
    print(f"No data found for feature '{FEATURE_NAME}' in CSV!")
else:
    print(f"Found {len(df_feature)} rows for feature '{FEATURE_NAME}'.")
    _cmap = plt.get_cmap('coolwarm')
    CLASS_COLORS = {cls: tuple(_cmap(cls / (NUM_BINS - 1))[:3]) for cls in range(NUM_BINS)}
    _, bins = pd.cut(df_feature['Chamfer_Distance_mm'], bins=NUM_BINS, retbins=True)
    df_feature['Error_Class'] = pd.cut(
        df_feature['Chamfer_Distance_mm'], bins=bins, labels=range(NUM_BINS), include_lowest=True
    )

valid_poses = []
valid_classes = []
for _, row in df_feature.iterrows():
    v_idx = int(row['Viewpoint'])
    if v_idx in poses_dict and pd.notna(row['Error_Class']):
        valid_poses.append(poses_dict[v_idx])
        valid_classes.append(int(row['Error_Class']))

colors_per_frame = [CLASS_COLORS[ec] for ec in valid_classes]
if not df_feature.empty:
    print(f"Chamfer Distance range: {df_feature['Chamfer_Distance_mm'].min():.2f} - {df_feature['Chamfer_Distance_mm'].max():.2f} mm")

# 4. SHOW RESULTS
if valid_poses:
    visualize_viewpoints_colored([workpiece_pcd, feature_pcd], valid_poses, colors_per_frame, axis_size=AXIS_SIZE)
else:
    print("No valid viewpoint poses to visualize.")

Sampling points from the whole workpiece mesh to create a PointCloud...
Sampling points from the feature mesh to create a PointCloud...


TypeError: paint_uniform_color(): incompatible function arguments. The following argument types are supported:
    1. (self: open3d.cpu.pybind.geometry.PointCloud, color: numpy.ndarray[numpy.float64[3, 1]]) -> open3d.cpu.pybind.geometry.PointCloud

Invoked with: PointCloud with 5000 points., [1.0, 0.0, 0.0]

### 6. Batch Processing (Multiple Workpieces)
Run the optimization pipeline automatically across an entire list of workpieces.

In [85]:
# ==========================================
# BATCH PROCESSING MULTIPLE WORKPIECES
# ==========================================
import os
import re
import json
import numpy as np
import pandas as pd
import open3d as o3d
import random
import time

# ------------------------------------------
# BATCH CONFIGURATION
# ------------------------------------------
# BATCH_WORKPIECES = ["TH0011AV"]  # Add as many as you want to test
BATCH_WORKPIECES = ["TH0011AV", "TH0012AV", "TH0021AV", "TH0022AV", "TH0031AV", "TH0032AV", "TH0041AV", "TH0042AV", "TH0051AV", "TH0052AV", "TH0061AV", "TH0062AV", "TH0071AV", "TH0072AV"]
EXPERIMENT = "test_8_simulation2"

USE_GROUND_TRUTH = False
# GT_CSV_PATH = f"processed_data/{EXPERIMENT}/metadata.csv"
CSV_PATH = "../../pointnet_pytorch_reflective/data/4_simulation/inference_results_pointnet2_moe.csv"

OPTIMIZATION_METHOD = "GRASP"
GRASP_ITERATIONS = 50
RCL_SIZE = 5

ALPHA = 0.3 # Coverability
BETA = 0.7 # Confidence
GAMMA = 0.5  # Submodular decay factor for Coverability
TOTAL_STEPS = 12
DISTANCE_THRESHOLD = 1.0

batch_results = []

# Load Inference CSV globally to save time
if USE_GROUND_TRUTH:
    df_inference = pd.read_csv(GT_CSV_PATH)
    df_inference.rename(columns={'filename': 'Filename', 'chamfer_value': 'Predicted_CD'}, inplace=True)
else:
    df_inference = pd.read_csv(CSV_PATH)
df_inference[['Parsed_Folder', 'Viewpoint', 'Feature']] = df_inference['Filename'].str.extract(r'(.*?)/viewpoint_simulated_(\d+)_(.*?)\.pcd')
df_inference['Viewpoint'] = df_inference['Viewpoint'].astype(int)

print(f"Starting batch optimization for {len(BATCH_WORKPIECES)} workpieces...")
print(f"Method: {OPTIMIZATION_METHOD} | Steps: {TOTAL_STEPS} | Alpha: {ALPHA} | Beta: {BETA} | GT: {USE_GROUND_TRUTH}\n")

for wp in BATCH_WORKPIECES:
    print(f"{'='*50}")
    print(f"Processing Workpiece: {wp}")
    print(f"{'='*50}")
    start_time = time.time()
    
    # --- 1. SET PATHS ---
    CAD_MODEL_DIR = wp
    GLOBAL_PCD_PATH = f"viewpoints_candidate/testing_data/{EXPERIMENT}/{wp}/pcd_all.pcd"
    GLOBAL_COVERED_JSON = f"viewpoints_candidate/testing_data/{EXPERIMENT}/{wp}/covered_indices.json"
    POSES_PATH = f"viewpoints_candidate/testing_data/{EXPERIMENT}/{wp}"
    
    # --- 2. LOAD DATA ---
    pcd_all = o3d.io.read_point_cloud(GLOBAL_PCD_PATH) if os.path.exists(GLOBAL_PCD_PATH) else o3d.geometry.PointCloud()
    if os.path.exists(GLOBAL_COVERED_JSON):
        with open(GLOBAL_COVERED_JSON, "r") as f:
            global_visibility_dict = json.load(f)
    else:
        global_visibility_dict = {}
        
    poses_dict = load_viewpoint_poses_dict(POSES_PATH)
    
    df_chamfer = df_inference[df_inference['Parsed_Folder'] == wp].copy()
    if df_chamfer.empty:
        print(f"Skipping {wp} - No data in CSV!")
        continue
        
    df_chamfer.rename(columns={'Predicted_CD': 'Chamfer_Distance_mm'}, inplace=True)
    features = df_chamfer['Feature'].unique()
    
    feature_indices = {}
    all_features_indices = set()
    for feat in features:
        feat_path = f"workpiece/{CAD_MODEL_DIR}/{feat}.stl"
        if os.path.exists(feat_path):
            f_mesh = o3d.io.read_triangle_mesh(feat_path)
            f_pcd = f_mesh.sample_points_poisson_disk(number_of_points=5000)
            dists = np.asarray(pcd_all.compute_point_cloud_distance(f_pcd))
            idx_set = set(np.where(dists < DISTANCE_THRESHOLD)[0])
            feature_indices[feat] = idx_set
            all_features_indices.update(idx_set)
            
    if len(all_features_indices) == 0:
        print(f"Skipping {wp} - No target feature points found!")
        continue

    # --- 3. RUN OPTIMIZATION ---
    if OPTIMIZATION_METHOD == "GREEDY":
        num_iterations = 1
        rcl_size = 1
    else:
        num_iterations = GRASP_ITERATIONS
        rcl_size = RCL_SIZE
        
    best_overall_sequence = []
    best_overall_score = -1.0
    best_point_coverage_counts = None
    
    # --- PRECOMPUTE STATIC CHAMFER DATA ---
    static_viewpoint_data = {}
    for v_idx, group in df_chamfer.groupby('Viewpoint'):
        v_idx = int(v_idx)
        if v_idx not in poses_dict or pd.isna(group['Chamfer_Distance_mm'].iloc[0]): continue
        camera_visible_40k = set(global_visibility_dict.get(str(v_idx), []))
        visible_target_indices = camera_visible_40k & all_features_indices
        if len(visible_target_indices) == 0: continue
        sum_weighted_chamfer = 0.0
        sum_points = 0
        for _, row in group.iterrows():
            feature = row['Feature']
            chamfer_dist = row['Chamfer_Distance_mm']
            if chamfer_dist < 0 or feature not in feature_indices: continue
            P_vj = len(camera_visible_40k & feature_indices[feature])
            sum_weighted_chamfer += (P_vj * chamfer_dist)
            sum_points += P_vj
        if sum_points == 0: continue
        weighted_chamfer = sum_weighted_chamfer / sum_points
        static_viewpoint_data[v_idx] = {
            'weighted_chamfer': weighted_chamfer,
            'visible_target_indices': visible_target_indices,
            'idx_arr': list(visible_target_indices)
        }
    
    for iteration in range(1, num_iterations + 1):
        point_coverage_counts = np.zeros(len(pcd_all.points))
        selected_viewpoints = []
        cumulative_utility = 0.0
        
        for k in range(1, TOTAL_STEPS + 1):
            db_records = []
            for v_idx, data in static_viewpoint_data.items():
                if v_idx in selected_viewpoints:
                    continue
                    
                idx_arr = data['idx_arr']
                c_vals = point_coverage_counts[idx_arr]
                submodular_sum = np.sum(GAMMA ** c_vals)
                coverability = submodular_sum / len(all_features_indices)
                
                db_records.append({
                    'Viewpoint': v_idx,
                    'Coverability': coverability,
                    'Chamfer_Distance': data['weighted_chamfer'],
                    '_visible_target_indices': data['visible_target_indices']
                })
                
            df_step = pd.DataFrame(db_records)
            if df_step.empty: break
            
            min_cov = df_step['Coverability'].min()
            max_cov = df_step['Coverability'].max()
            df_step['Norm_Coverability'] = (df_step['Coverability'] - min_cov) / (max_cov - min_cov) if max_cov > min_cov else 1.0
            
            df_step['Uncertainty'] = (1.0 * df_step['Norm_Coverability']) + (1.0 * df_step['Chamfer_Distance'])
            min_uncert = df_step['Uncertainty'].min()
            max_uncert = df_step['Uncertainty'].max()
            df_step['Confidence'] = 1.0 - ((df_step['Uncertainty'] - min_uncert) / (max_uncert - min_uncert)) if max_uncert > min_uncert else 1.0
            
            df_step['Information_Gain'] = df_step['Norm_Coverability']
            df_step['Utility_Score'] = ALPHA * df_step['Information_Gain'] + BETA * df_step['Confidence']
            
            df_step = df_step.sort_values(by='Utility_Score', ascending=False).reset_index(drop=True)
            
            actual_rcl_size = min(rcl_size, len(df_step))
            rcl = df_step.head(actual_rcl_size)
            chosen_idx = random.randint(0, actual_rcl_size - 1)
            best_row = rcl.iloc[chosen_idx]
            
            cumulative_utility += best_row['Utility_Score']
            selected_viewpoints.append(int(best_row['Viewpoint']))
            
            for idx in best_row['_visible_target_indices']:
                point_coverage_counts[idx] += 1
                
        if cumulative_utility > best_overall_score:
            best_overall_score = cumulative_utility
            best_overall_sequence = selected_viewpoints
            best_point_coverage_counts = point_coverage_counts.copy()
        if OPTIMIZATION_METHOD == "GRASP":
            print(f"  Iteration {iteration}/{num_iterations} | Current Utility: {cumulative_utility:.4f} | Best Sum Utility: {best_overall_score:.4f}")
            
    process_time = time.time() - start_time
    points_covered = np.sum(best_point_coverage_counts > 0)
    total_points = len(all_features_indices)
    
    print(f"--> {wp} Done! | Utility: {best_overall_score:.4f} | Covered: {points_covered}/{total_points} | Time: {process_time:.1f}s")
    print(f"--> Best Sequence: {best_overall_sequence}\n")
    
    batch_results.append({
        'Workpiece': wp,
        'Best_Utility_Score': best_overall_score,
        'Points_Covered': f"{points_covered} / {total_points}",
        'Coverage_Percentage': f"{(points_covered / total_points * 100):.1f}%",
        'Optimal_Sequence': str(best_overall_sequence),
        'Processing_Time_s': process_time
    })

if batch_results:
    df_results = pd.DataFrame(batch_results)
    print("\n\n" + "="*60)
    print("BATCH OPTIMIZATION SUMMARY")
    print("="*60)
    display(df_results)


Starting batch optimization for 14 workpieces...
Method: GRASP | Steps: 12 | Alpha: 0.3 | Beta: 0.7 | GT: False

Processing Workpiece: TH0011AV
  Iteration 1/50 | Current Utility: 8.3193 | Best Sum Utility: 8.3193
  Iteration 2/50 | Current Utility: 8.2870 | Best Sum Utility: 8.3193
  Iteration 3/50 | Current Utility: 8.3757 | Best Sum Utility: 8.3757
  Iteration 4/50 | Current Utility: 8.4138 | Best Sum Utility: 8.4138
  Iteration 5/50 | Current Utility: 8.3091 | Best Sum Utility: 8.4138
  Iteration 6/50 | Current Utility: 8.3658 | Best Sum Utility: 8.4138
  Iteration 7/50 | Current Utility: 8.2917 | Best Sum Utility: 8.4138
  Iteration 8/50 | Current Utility: 8.3919 | Best Sum Utility: 8.4138
  Iteration 9/50 | Current Utility: 8.2549 | Best Sum Utility: 8.4138
  Iteration 10/50 | Current Utility: 8.3577 | Best Sum Utility: 8.4138
  Iteration 11/50 | Current Utility: 8.3715 | Best Sum Utility: 8.4138
  Iteration 12/50 | Current Utility: 8.3530 | Best Sum Utility: 8.4138
  Iteration 1

,Workpiece,Best_Utility_Score,Points_Covered,Coverage_Percentage,Optimal_Sequence,Processing_Time_s
0,TH0011AV,8.413760,10492 / 13095,80.1%,"[286, 285, 430, 429, 142, 269, 413, 270, 397, ...",163.555151
1,TH0012AV,8.467970,11517 / 15682,73.4%,"[281, 425, 137, 426, 138, 378, 362, 90, 346, 2...",171.485791
2,TH0021AV,8.381302,9657 / 11919,81.0%,"[280, 424, 408, 120, 264, 425, 248, 136, 392, ...",153.536177
3,TH0022AV,8.324655,10288 / 13897,74.0%,"[210, 138, 66, 209, 211, 137, 67, 139, 127, 19...",81.189277
4,TH0031AV,8.402372,11135 / 12527,88.9%,"[211, 127, 199, 187, 55, 115, 43, 103, 175, 16...",82.799833
5,TH0032AV,8.396549,9881 / 14819,66.7%,"[424, 264, 408, 280, 136, 120, 137, 281, 425, ...",166.902175
6,TH0041AV,8.394929,9781 / 12122,80.7%,"[136, 408, 280, 137, 264, 424, 120, 409, 248, ...",150.807521
7,TH0042AV,8.395162,9252 / 14214,65.1%,"[281, 425, 408, 120, 264, 424, 392, 137, 265, ...",156.364414
8,TH0051AV,8.396928,9938 / 12313,80.7%,"[424, 136, 280, 264, 281, 408, 120, 137, 121, ...",150.638215
9,TH0052AV,8.378061,9768 / 14548,67.1%,"[408, 280, 424, 264, 136, 120, 423, 248, 392, ...",160.348430
